In [ ]:
import sys, platform, importlib, subprocess, socket, glob
print("python:", sys.version.split()[0], "|", platform.platform())

print("\n--- can we import the libraries we need? ---")
for m in ["rdkit","lightgbm","xgboost","catboost","sklearn","scipy","joblib","pandas","numpy"]:
    try:
        mod = importlib.import_module(m)
        print(f"  {m:10s} OK   {getattr(mod,'__version__','?')}")
    except Exception as e:
        print(f"  {m:10s} MISSING ({type(e).__name__})")

print("\n--- is rdkit installed anywhere on disk? ---")
hits = glob.glob("/opt/conda/lib/python*/site-packages/rdkit*") \
     + glob.glob("/usr/local/lib/python*/dist-packages/rdkit*") \
     + glob.glob("/usr/lib/python3/dist-packages/rdkit*")
print("  ", hits[:5] if hits else "not found on disk")

print("\n--- does the kernel actually have internet? ---")
try:
    socket.setdefaulttimeout(8)
    ip = socket.gethostbyname("pypi.org")
    print("  DNS pypi.org ->", ip)
    socket.create_connection(("pypi.org", 443), timeout=8).close()
    print("  HTTPS to pypi.org: OK  -> internet IS enabled")
except Exception as e:
    print(f"  NO INTERNET: {type(e).__name__}: {e}")

print("\n--- pip list (chem-related) ---")
try:
    out = subprocess.run([sys.executable,"-m","pip","list"],capture_output=True,text=True,timeout=120).stdout
    print("  " + "\n  ".join(l for l in out.splitlines()
                              if any(k in l.lower() for k in ("rdkit","chem","mol","boost","learn"))))
except Exception as e:
    print("  pip list failed:", e)

print("\n--- competition data visible? ---")
print("  ", glob.glob("/kaggle/input/*"))